# Building pseudonymous identifiers

This notebook implements the technique described in
[Anonymisation and safe preparation](anonymisation.md): turning identifying
information into a pseudonymous key that two institutions can compare without
either of them sending the other a list of names.

It then does something that matters more than the technique: it **measures what
the technique costs you**. Hashing forces exact matching, and exact matching on
imperfect data misses matches. By the end you will have a number for how many.

**What you will do**

1. Rebuild the prepared data from chapter 2.2
2. Hash a string, and see why one character changes everything
3. See why a naive hash of a low-entropy field protects nothing
4. Add a secret pepper and build a pseudonymous identifier
5. Link the two registers on that identifier, and measure the result against
   the known truth
6. Produce a shareable extract and an internal linkage key

## 0. Setup and prepared data

Every notebook in this toolkit is self-contained, so this cell repeats the
cleaning from [chapter 2.2](nb01-inspect-and-prepare.ipynb). It is the same code,
unchanged.

In [1]:
import hashlib
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

DATA = Path("../../data")
if not DATA.exists():
    DATA = Path("data")

fonasa = pd.read_csv(DATA / "fonasa_sample.csv", dtype=str)
suseso = pd.read_csv(DATA / "suseso_sample.csv", dtype=str)


def basic_text_clean(series):
    return (series.astype("string").str.strip().str.upper()
            .str.replace(r"\s+", " ", regex=True))


def remove_accents(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(c for c in value if not unicodedata.combining(c))


def standardise_name(series):
    cleaned = basic_text_clean(series)
    cleaned = cleaned.map(remove_accents, na_action="ignore").astype("string")
    cleaned = cleaned.str.replace(r"[^A-ZN ]", "", regex=True)
    return cleaned.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_sex(series):
    cleaned = basic_text_clean(series)
    return cleaned.replace({"HOMBRE": "M", "MUJER": "F", "MASCULINO": "M",
                            "FEMENINO": "F", "": pd.NA})


for df in (fonasa, suseso):
    for col in ["nombre", "ap1", "ap2"]:
        df[f"{col}_clean"] = standardise_name(df[col])
    df["sexo_clean"] = clean_sex(df["sexo"])
    df["nac_clean"] = basic_text_clean(df["nacionalidad"])

print(f"fonasa: {len(fonasa):,} records | suseso: {len(suseso):,} records")

fonasa: 30,000 records | suseso: 27,000 records


## 1. Hashing in one cell

A cryptographic hash function takes any input and returns a fixed-length string.
The same input always gives the same output. Different inputs give outputs that
look unrelated, and there is no practical way to run the function backwards.

In [2]:
def sha256(text):
    return hashlib.sha256(text.encode()).hexdigest()


print(sha256("JUAN PEREZ GOMEZ"))
print(sha256("JUAN PEREZ GOMES"))   # one character different

32640349596ef03130b86f6d6a53cf2d2d2b3687b26ad97b4fd2a6432ace7656
0a0d50318156a7b34921d379f4ce303ebd7f3214560ddf7bbdc7888057983813


That is the **avalanche effect**: a single character changes the entire output.

It is exactly the property that makes hashing useful for identity — and exactly
the property that makes it brittle for linkage. Two records for the same person
that differ by one typographical error produce two completely unrelated hashes.
There is no such thing as an approximate hash match. This is the central
limitation of the whole approach, and section 5 measures it.

## 2. Why a naive hash protects nothing

A hash is not encryption and it is not anonymisation. If the set of possible
inputs is small, anyone can hash all of them and build a lookup table.

Watch how little protection an unsalted hash of a low-entropy field gives.

In [3]:
# An attacker who suspects the field is sex simply tries the plausible values
for guess in ["M", "F"]:
    print(f"{guess} -> {sha256(guess)}")

print()
print("Hashed sex values actually present in the register:")
print(fonasa["sexo_clean"].dropna().map(sha256).value_counts().to_string())

M -> 08f271887ce94707da822d5263bae19d5519cb3614e0daedc4c7ce5dab7473f1
F -> f67ab10ad4e4c53121b6a5fe4da9c10ddee905b978d3788d2723d7bfacbe28a9

Hashed sex values actually present in the register:
sexo_clean
08f271887ce94707da822d5263bae19d5519cb3614e0daedc4c7ce5dab7473f1    14745
f67ab10ad4e4c53121b6a5fe4da9c10ddee905b978d3788d2723d7bfacbe28a9    14308


Two distinct hashes, matching the two guesses exactly. The hash has been
reversed by anyone willing to spend a second on it.

Names are a larger space, but not a *large* one: national name frequency lists
are public, and an attacker with a candidate list can hash every entry and
compare. This is a dictionary attack, and it is entirely practical against
hashed personal data.

The defence is a **secret** added to the input before hashing, so that an
attacker without the secret cannot precompute anything. Added per-record it is
called a salt; added as a single shared constant it is usually called a
**pepper**. For linkage it must be a pepper: both institutions have to use the
identical secret, or their hashes will never agree.

In [4]:
# TRAINING VALUE ONLY. In production this is never a literal in code.
SECRET_PEPPER = "change-me-and-keep-me-secret"


def make_pseudo_id(key_string, pepper=SECRET_PEPPER):
    """Hash a canonical key string together with a secret pepper."""
    if key_string is None or pd.isna(key_string) or str(key_string).strip() == "":
        return None
    return hashlib.sha256(f"{pepper}::{key_string}".encode()).hexdigest()


print("with pepper   :", make_pseudo_id("M"))
print("without pepper:", sha256("M"))

with pepper   : 6ff59b1781191fba8efe2e26b062b9c18c78173869b67a661fb866213c4eb8b9
without pepper: 08f271887ce94707da822d5263bae19d5519cb3614e0daedc4c7ce5dab7473f1


The pepper is the whole security of the scheme. Three rules:

- **Never** in code, never in a notebook, never in version control. Read it from
  an environment variable or a secrets vault.
- Shared with the other institution through a channel separate from the data.
- If it leaks, every pseudonymous identifier ever produced with it is
  compromised, and you must re-issue with a new one.

## 3. The key string

What you hash matters as much as how you hash it. The input is a **canonical key
string**: the identifying fields, cleaned, in a fixed order, with a fixed
separator.

Two decisions have to be made explicitly, and both change the results.

**Which fields are required.** If a required field is missing, the record gets
no identifier and cannot be linked at all.

**How missing optional fields are represented.** An empty string is not the same
as an absent field, and the two institutions must make the same choice.

In [5]:
def make_key_string(nombre, ap1, ap2, sexo, nac):
    """Canonical key string. Required: nombre, ap1, sexo, nac. Optional: ap2."""
    required = [nombre, ap1, sexo, nac]
    if any(pd.isna(p) or str(p).strip() == "" for p in required):
        return None
    ap2_part = "" if (pd.isna(ap2) or str(ap2).strip() == "") else str(ap2)
    return "|".join([str(nombre), str(ap1), ap2_part, str(sexo), str(nac)])


example = make_key_string("JUAN", "PEREZ", "GOMEZ", "M", "CHILENA")
print("key string:", example)
print("pseudo_id :", make_pseudo_id(example))
print()
# Same person, second surname not recorded
example_2 = make_key_string("JUAN", "PEREZ", None, "M", "CHILENA")
print("key string:", example_2)
print("pseudo_id :", make_pseudo_id(example_2))

key string: JUAN|PEREZ|GOMEZ|M|CHILENA
pseudo_id : aaddcd5aadbb165aa88aa5688dbe6d985981768fd380100050945a9540d03bec

key string: JUAN|PEREZ||M|CHILENA
pseudo_id : 1bd837b8648cefcc776ec34accf7f560aa0259fadb654dae14555387c449e4d0


Those two records describe the same person, and they produce different
identifiers. Choosing to treat a missing `ap2` as an empty string rather than
excluding the record is a decision to accept that cost in exchange for keeping
the record linkable at all — but the cost is real, and it is invisible unless
you write it down.

In [6]:
for df in (fonasa, suseso):
    df["key_string"] = [
        make_key_string(n, a1, a2, s, nc)
        for n, a1, a2, s, nc in zip(
            df["nombre_clean"], df["ap1_clean"], df["ap2_clean"],
            df["sexo_clean"], df["nac_clean"],
        )
    ]
    df["pseudo_id"] = df["key_string"].map(make_pseudo_id)

for name, df in [("fonasa", fonasa), ("suseso", suseso)]:
    n_ok = df["pseudo_id"].notna().sum()
    print(f"{name}: {n_ok:,} of {len(df):,} records received a pseudo_id "
          f"({100*n_ok/len(df):.2f}%) - {len(df)-n_ok:,} could not be built")

fonasa: 25,278 of 30,000 records received a pseudo_id (84.26%) - 4,722 could not be built
suseso: 23,415 of 27,000 records received a pseudo_id (86.72%) - 3,585 could not be built


The records that received no identifier are those missing at least one required
field. They are not linkable by this method at all, and there are more of them
than the per-field missingness rates would suggest, because a record only needs
to be missing *one* required field to drop out.

Note what has happened: a privacy mechanism has just excluded a specific
subpopulation from the analysis. Whether those people differ from the rest is
now a question you have to answer.

## 4. Linking on the pseudonymous identifier

Two institutions can now exchange files containing `pseudo_id` and non-identifying
attributes, and link them with an ordinary join. Neither has to send the other a
name.

In [7]:
left = fonasa.dropna(subset=["pseudo_id"])[["unique_id", "pseudo_id", "true_person_id"]]
right = suseso.dropna(subset=["pseudo_id"])[["unique_id", "pseudo_id", "true_person_id"]]

linked = left.merge(right, on="pseudo_id", suffixes=("_f", "_s"))

print(f"linked pairs: {len(linked):,}")
print(f"match rate against the health register: {100*len(linked)/len(fonasa):.2f}%")

linked pairs: 1,080
match rate against the health register: 3.60%


## 5. What did it cost?

The bundled data is synthetic, so we know the answer: exactly 4,500 people
appear in both registers. That lets us do what you normally cannot, and score
the method.

In [8]:
TRUE_MATCHES = 4500

true_positives = (linked["true_person_id_f"] == linked["true_person_id_s"]).sum()
false_positives = len(linked) - true_positives

precision = true_positives / len(linked) if len(linked) else float("nan")
recall = true_positives / TRUE_MATCHES

print(f"pairs proposed        : {len(linked):,}")
print(f"correct (true matches): {true_positives:,}")
print(f"incorrect             : {false_positives:,}")
print()
print(f"precision : {precision:.4f}   (of the pairs proposed, this share are right)")
print(f"recall    : {recall:.4f}   (of the {TRUE_MATCHES:,} real matches, this share were found)")
print(f"missed    : {TRUE_MATCHES - true_positives:,} true matches never proposed")

pairs proposed        : 1,080
correct (true matches): 1,080
incorrect             : 0

precision : 1.0000   (of the pairs proposed, this share are right)
recall    : 0.2400   (of the 4,500 real matches, this share were found)
missed    : 3,420 true matches never proposed


This is the honest result, and it is the reason the rest of this toolkit exists.

Precision is 1.0000: every single pair it proposed was correct. When five
cleaned identifying fields agree exactly, the records really do describe the same
person. Do not over-read this — in a register of tens of millions, some
coincidental agreements on five fields would occur, so precision would be high
but not exactly one. In a file of this size it is.

Recall is not. The great majority of the true matches were never found, and they
were lost for two reasons, both structural:

1. Records with a missing required field never received an identifier.
2. Any difference at all in any field — one letter, one accent that survived
   cleaning, an abbreviated name, a second surname recorded in one system and
   not the other — produces two unrelated hashes.

No amount of care in the hashing fixes this. The method cannot tolerate error,
and the data contains error. That is the ceiling.

## 6. Two extracts, not one

The separation principle in practice: produce two files with different audiences.

The **shareable extract** carries the pseudonymous key and non-identifying
attributes. It can cross an institutional boundary.

The **internal linkage key** keeps the mapping from `pseudo_id` back to the
record and its identifiers. It stays inside the institution that produced it,
and it is what lets you attach results back to your own records afterwards.

In [9]:
shareable = (
    fonasa.dropna(subset=["pseudo_id"])
    .assign(source="fonasa")[["pseudo_id", "source", "sexo_clean", "nac_clean"]]
)

internal = (
    fonasa.dropna(subset=["pseudo_id"])
    [["pseudo_id", "unique_id", "nombre", "ap1", "ap2",
      "nombre_clean", "ap1_clean", "ap2_clean", "sexo_clean", "nac_clean"]]
)

print(f"shareable extract : {shareable.shape[0]:,} rows x {shareable.shape[1]} cols "
      f"-> {list(shareable.columns)}")
print(f"internal key      : {internal.shape[0]:,} rows x {internal.shape[1]} cols  (never leaves the institution)")
shareable.head(3)

shareable extract : 25,278 rows x 4 cols -> ['pseudo_id', 'source', 'sexo_clean', 'nac_clean']
internal key      : 25,278 rows x 10 cols  (never leaves the institution)


,pseudo_id,source,sexo_clean,nac_clean
1,777450775788fc1adf0d8fa5af7f7f7790a07e077dfce2...,fonasa,M,CHILENA
2,d9974ef35e6963efc5633e9fde7a6980fa609c7323f502...,fonasa,F,CHILENA
3,d02295f8426c9128c422593a19f68a43cd464e86fe42e6...,fonasa,M,CHILENA


Note that even the shareable extract is **pseudonymous, not anonymous**. Every
row still refers to one identifiable person, and anyone holding the pepper and a
candidate list can test whether a given person is present. It reduces risk
substantially; it does not eliminate it, and it should be handled under the same
governance as personal data.

## 7. Documenting it

A pseudonymisation scheme is only reproducible if these details are recorded.
Both institutions need to hold the identical specification, or their identifiers
will not agree.

In [10]:
pd.DataFrame({
    "item": [
        "Algorithm",
        "Key string fields, in order",
        "Field separator",
        "Required fields",
        "Missing optional field",
        "Secret",
        "Records with an identifier (health register)",
        "Records with an identifier (social security register)",
        "Linked pairs",
        "Precision against known truth",
        "Recall against known truth",
    ],
    "value": [
        "SHA-256 with a shared secret pepper",
        "nombre_clean | ap1_clean | ap2_clean | sexo_clean | nac_clean",
        "| (pipe)",
        "nombre, ap1, sexo, nacionalidad",
        "represented as an empty string",
        "shared out of band; never stored with the data",
        f"{fonasa['pseudo_id'].notna().sum():,} ({100*fonasa['pseudo_id'].notna().mean():.2f}%)",
        f"{suseso['pseudo_id'].notna().sum():,} ({100*suseso['pseudo_id'].notna().mean():.2f}%)",
        f"{len(linked):,}",
        f"{precision:.4f}",
        f"{recall:.4f}",
    ],
})

,item,value
0,Algorithm,SHA-256 with a shared secret pepper
1,"Key string fields, in order",nombre_clean | ap1_clean | ap2_clean | sexo_cl...
2,Field separator,| (pipe)
3,Required fields,"nombre, ap1, sexo, nacionalidad"
4,Missing optional field,represented as an empty string
5,Secret,shared out of band; never stored with the data
6,Records with an identifier (health register),"25,278 (84.26%)"
7,Records with an identifier (social security re...,"23,415 (86.72%)"
8,Linked pairs,"1,080"
9,Precision against known truth,1.0000


## Where this goes next

You have a working, privacy-preserving linkage with excellent precision and poor
recall. For some purposes that is enough: if you only need pairs you are certain
about, and you can live with finding a minority of them, stop here.

For most statistical purposes it is not enough, because the records you failed to
link are not a random sample of the population.

The rest of Section 2 is about recovering those matches: comparing records that
*nearly* agree rather than requiring them to agree exactly.
[Chapter 2.4](linkage-approaches.md) starts with deterministic rules that
tolerate specified kinds of disagreement, and then with the framework that scores
partial agreement rather than demanding it.